# Charging points, households and income per municipality

This notebook merges the public charging point data with the CBS household and income data, and saves the result for the other notebooks.

**Input files (`data/raw/`):**
- `laadpunten_per_gemeente.csv`: regular public charging points per municipality, July 2026 (NAL)
- `inkomens_per_gemeente.csv`: private households and standardised income per municipality, 2024* (CBS 86161NED)

**Output (`data/processed/`):**
- `chargingpoints_households_income.csv`: one row per municipality (342)

In [1]:
import pandas as pd

In [2]:
# Load the data
charging = pd.read_csv("../data/raw/laadpunten_per_gemeente.csv", sep=";")
charging.columns = ["municipality", "charging_points"]

income = pd.read_csv("../data/raw/inkomens_per_gemeente.csv", sep=";", encoding="utf-8-sig")
income = income.rename(columns={"Regio's": "municipality"})

# Numbers are stored as text with leading spaces -> convert to numbers
for col in income.columns[4:]:
    income[col] = pd.to_numeric(income[col].astype(str).str.strip(), errors="coerce")

# Align names: CBS name -> name used in the charging point file
name_mapping = {
    "'s-Gravenhage (gemeente)": "Den Haag",
    "Groningen (gemeente)": "Groningen",
    "Utrecht (gemeente)": "Utrecht",
    "Beek (L.)": "Beek",
    "Stein (L.)": "Stein",
    "Hengelo (O.)": "Hengelo",
    "Laren (NH.)": "Laren",
    "Middelburg (Z.)": "Middelburg",
    "Rijswijk (ZH.)": "Rijswijk",
}
income["municipality"] = income["municipality"].replace(name_mapping)

# Remove the national total row
charging = charging[charging["municipality"] != "Totaal"]

# Merge
data = charging.merge(income, on="municipality", how="inner")

# Keep the relevant columns and give them English names
data = data.rename(columns={
    "Particuliere huishoudens (x 1 000)": "households_x1000",
    "Inkomen/Gemiddeld gestandaardiseerd inkomen (1 000 euro)": "mean_standardised_income_keur",
})
data = data[["municipality", "charging_points", "households_x1000", "mean_standardised_income_keur"]]
print(data.shape)  # (342, 4)

# Save
data.to_csv("../data/processed/chargingpoints_households_income.csv", index=False)

(342, 4)


In [3]:
data.head()

,municipality,charging_points,households_x1000,mean_standardised_income_keur
0,Aa en Hunze,89,11.4,42.6
1,Aalsmeer,316,14.2,52.1
2,Aalten,138,11.7,40.5
3,Achtkarspelen,54,12.0,36.3
4,Alblasserdam,183,8.5,41.8
